#### ENRICH CUSTOMERS FROM BRONZE TO SILVER

In [0]:
cust_df = spark.table('ORDERS_SALES.BRONZE.CUSTOMERS')
cust_df.limit(1).display()

In [0]:
from pyspark.sql.functions import *

cust_enriched_df = cust_df.select(
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    upper(trim(col("customer_city"))).alias("customer_city"),
    upper(trim(col("customer_state"))).alias("customer_state"),
    "file_name",
    "file_path",
    current_timestamp().alias('load_timestamp')
).filter('customer_id IS NOT NULL')

display(cust_enriched_df.limit(1))

In [0]:
cust_enriched_df.createOrReplaceTempView('enriched_customers_temp_vw')

#### INCREMENTAL LOAD `ORDERS_SALES.SILVER.CUSTOMERS_ENRICHED`

In [0]:
%sql
MERGE INTO ORDERS_SALES.SILVER.CUSTOMERS_ENRICHED tgt
USING enriched_customers_temp_vw src
ON (src.customer_id = tgt.customer_id)
WHEN MATCHED
    THEN UPDATE SET
    tgt.customer_id = src.customer_id,
    tgt.customer_unique_id = src.customer_unique_id,
    tgt.customer_zip_code_prefix = src.customer_zip_code_prefix,
    tgt.customer_city = src.customer_city,
    tgt.customer_state = src.customer_state
WHEN NOT MATCHED THEN
INSERT (customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state, origin_file_path, load_timestamp)
VALUES (src.customer_id, src.customer_unique_id, src.customer_zip_code_prefix, src.customer_city, src.customer_state, src.file_path, current_timestamp)

In [0]:
dbutils.notebook.exit('SUCCESSFULLY LOADED INTO ORDERS_SALES.SILVER.CUSTOMERS_ENRICHED')